<a href="https://colab.research.google.com/github/esdrasfelipe07/resolvedor-sudoku-csp/blob/main/Sudoku.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sudoku


In [ ]:
from copy import deepcopy

def eh_valido(tabuleiro, linha, coluna, numero):
    bloco_linha, bloco_coluna = 3 * (linha // 3), 3 * (coluna // 3)
    for i in range(9):
        if tabuleiro[linha][i] == numero or tabuleiro[i][coluna] == numero:
            return False
    for i in range(3):
        for j in range(3):
            if tabuleiro[bloco_linha + i][bloco_coluna + j] == numero:
                return False
    return True

def calcular_dominios(tabuleiro):
    dominios = {}
    for linha in range(9):
        for coluna in range(9):
            if tabuleiro[linha][coluna] == 0:
                possiveis = []
                for numero in range(1, 10):
                    if eh_valido(tabuleiro, linha, coluna, numero):
                        possiveis.append(numero)
                dominios[(linha, coluna)] = possiveis
    return dominios

def selecionar_celula_indefinida(dominios):
    return min(dominios, key=lambda k: len(dominios[k]), default=None)

def verificacao_antecipada(dominios, linha, coluna, valor):
    novos_dominios = deepcopy(dominios)
    del novos_dominios[(linha, coluna)]
    for (l, c) in novos_dominios:
        if l == linha or c == coluna or (l // 3 == linha // 3 and c // 3 == coluna // 3):
            if valor in novos_dominios[(l, c)]:
                novos_dominios[(l, c)].remove(valor)
                if not novos_dominios[(l, c)]:
                    return None
    return novos_dominios

def resolver(tabuleiro):
    dominios = calcular_dominios(tabuleiro)
    return retroceder(tabuleiro, dominios)

def retroceder(tabuleiro, dominios):
    if not dominios:
        return True
    celula = selecionar_celula_indefinida(dominios)
    if not celula:
        return False
    linha, coluna = celula
    for valor in dominios[celula]:
        if eh_valido(tabuleiro, linha, coluna, valor):
            tabuleiro[linha][coluna] = valor
            novos_dominios = verificacao_antecipada(dominios, linha, coluna, valor)
            if novos_dominios is not None and retroceder(tabuleiro, novos_dominios):
                return True
            tabuleiro[linha][coluna] = 0
    return False

def imprimir_tabuleiro(tabuleiro):
    for i in range(9):
        if i % 3 == 0 and i != 0:
            print("-" * 21)
        for j in range(9):
            if j % 3 == 0 and j != 0:
                print("| ", end="")
            print(tabuleiro[i][j] if tabuleiro[i][j] != 0 else ".", end=" ")
        print()

def tabuleiro_valido(tabuleiro):
    for linha in range(9):
        for coluna in range(9):
            numero = tabuleiro[linha][coluna]
            if numero != 0:
                tabuleiro[linha][coluna] = 0
                if not eh_valido(tabuleiro, linha, coluna, numero):
                    tabuleiro[linha][coluna] = numero
                    return False
                tabuleiro[linha][coluna] = numero
    return True

def inserir_tabuleiro():
    print("\nInsira o tabuleiro do Sudoku:")
    print("→ Digite 9 linhas, cada uma com 9 números de 0 a 9 separados por espaço.")
    print("→ Use o número 0 para indicar células vazias.\n")

    tabuleiro = []
    for i in range(9):
        while True:
            try:
                linha = input(f"Linha {i+1}: ").strip()
                valores = list(map(int, linha.split()))
                if len(valores) != 9 or any(num < 0 or num > 9 for num in valores):
                    raise ValueError
                tabuleiro.append(valores)
                break
            except ValueError:
                print(" Entrada inválida. Digite exatamente 9 números entre 0 e 9 separados por espaço.")

    if not tabuleiro_valido(tabuleiro):
        print(" O tabuleiro inicial contém conflitos com as regras do Sudoku.")
        return inserir_tabuleiro()

    return tabuleiro

def preencher_restante(tabuleiro_base):
    print("\nAgora complete o restante do tabuleiro (use 0 para deixar vazio):")
    tabuleiro = deepcopy(tabuleiro_base)
    for i in range(9):
        while True:
            try:
                entrada = input(f"Linha {i+1} (atual: {' '.join(map(str, tabuleiro_base[i]))}): ").strip()
                valores = list(map(int, entrada.split()))
                if len(valores) != 9 or any(n < 0 or n > 9 for n in valores):
                    raise ValueError
                for j in range(9):
                    if tabuleiro_base[i][j] != 0 and valores[j] != tabuleiro_base[i][j]:
                        print(" Você não pode alterar os valores fixos.")
                        raise ValueError
                tabuleiro[i] = valores
                break
            except ValueError:
                print(" Entrada inválida. Certifique-se de inserir 9 números entre 0 e 9 e não alterar os valores já definidos.")
    if not tabuleiro_valido(tabuleiro):
        print(" O tabuleiro contém conflitos. Vamos tentar novamente.\n")
        return preencher_restante(tabuleiro_base)
    return tabuleiro

def mostrar_regras_sudoku():
    print("\nRegras do Sudoku:")
    print("- O tabuleiro é uma grade de 9x9, dividida em 9 blocos de 3x3.")
    print("- O objetivo é preencher todas as células com números de 1 a 9.")
    print("- Cada linha deve conter todos os números de 1 a 9, sem repetições.")
    print("- Cada coluna também deve conter todos os números de 1 a 9, sem repetições.")
    print("- Cada um dos nove blocos 3x3 também deve conter todos os números de 1 a 9, sem repetições.")

def escolher_opcao():
    print("\nEscolha uma opção para começar:")
    print("1 - Digitar tabuleiro totalmente vazio")
    print("2 - Usar nível fácil e completar os valores")
    print("3 - Usar nível difícil e completar os valores")
    escolha = input("Digite 1, 2 ou 3: ")

    if escolha == "2":
        tabuleiro_facil = [
            [5, 3, 0, 0, 7, 0, 0, 0, 0],
            [6, 0, 0, 1, 9, 5, 0, 0, 0],
            [0, 9, 8, 0, 0, 0, 0, 6, 0],
            [8, 0, 0, 0, 6, 0, 0, 0, 3],
            [4, 0, 0, 8, 0, 3, 0, 0, 1],
            [7, 0, 0, 0, 2, 0, 0, 0, 6],
            [0, 6, 0, 0, 0, 0, 2, 8, 0],
            [0, 0, 0, 4, 1, 9, 0, 0, 5],
            [0, 0, 0, 0, 8, 0, 0, 7, 9]
        ]
        mostrar_regras_sudoku()
        return preencher_restante(tabuleiro_facil)

    elif escolha == "3":
        tabuleiro_dificil = [
            [0, 0, 0, 0, 0, 0, 0, 1, 2],
            [0, 0, 0, 0, 0, 3, 0, 0, 0],
            [0, 0, 1, 0, 9, 0, 0, 0, 0],
            [0, 0, 0, 5, 0, 7, 0, 0, 0],
            [0, 0, 4, 0, 0, 0, 1, 0, 0],
            [0, 9, 0, 0, 0, 0, 0, 8, 0],
            [0, 0, 0, 0, 7, 0, 3, 6, 0],
            [0, 0, 0, 6, 0, 0, 0, 0, 0],
            [5, 7, 0, 0, 0, 0, 0, 0, 0]
        ]
        mostrar_regras_sudoku()
        return preencher_restante(tabuleiro_dificil)

    else:
        print("\nVocê escolheu um tabuleiro vazio. Vamos preenchê-lo:")
        mostrar_regras_sudoku()
        return inserir_tabuleiro()

def main():
    tabuleiro = escolher_opcao()
    print("\nTabuleiro inicial:")
    imprimir_tabuleiro(tabuleiro)

    if resolver(tabuleiro):
        print("\nSudoku resolvido com sucesso:")
        imprimir_tabuleiro(tabuleiro)
    else:
        print("\nNão foi possível resolver o Sudoku com o tabuleiro fornecido.")

if __name__ == "__main__":
    main()



Escolha uma opção para começar:
1 - Digitar tabuleiro totalmente vazio
2 - Usar nível fácil e completar os valores
3 - Usar nível difícil e completar os valores
